## Deploying Flixtube to a Local Kubernetes Cluster

In this notebook, we look at how we can deploy the Flixtube microservice application to Minikube's Kubernetes cluster.

- Make sure you have a Minikube Kubernetes cluster running, e.g. `minukube start`.
- Also make sure you have installed the `kubectl` (and `docker`) tool on your computer.

---

## Architecture

The `Flixtube` application is a microservice-based application with seven custom microservices:
  - `Web`: A Blazor frontend service with a user interface.
  - `Gateway`: An API gateway service, which the `Web` service talks to, and which redirects requests to the various other services below.
  - `Metadata`: A metadata service that manages video metadata (e.g. the name of a video).
  - `History`: A history service that manages video viewing history (e.g. the name, date and time of when a video was played).
  - `VideoUpload`: A service used to upload a video from the `Web` frontend via the `Gateway` to the `VideoStorage` service.
  - `VideoStorage`: A service responsible for storing video files.
  - `Videostreaming`: A service used to stream a video from the `VideoStorage` service via the `Gateway` to the `Web` service.

The `Flixtube` application uses a number of **backing services**:
  - `Sql Server`: An SQL Server service to store relational data.
  - `Minio` or `Azure Storage`: A Minio or Azure Storage service to store Blob data (video files).
  - `Rabbit MQ`: A Rabbit Message Queue (MQ) service for publisher/subscriber-based messaging.

<img src="notebook_images/flixtube_architecture.drawio.png"  width="800" alt="Flixtube Architecture" />

**Communication**

- The **Browser** communicates with:
  - The `Web` microservice via HTTP.
  - The `Gateway` microservice via HTTP when streaming (playing) a Video.
- The `Gateway` communicates with the microservices below via HTTP:
  - `Web`, `Metadata`, `History`, `VideoStreaming`, `VideoUpload`, and `VideoStorage`.
- `VideoStorage` communicates with:
  - The `Gateway`, `VideoStreaming`, and `VideoUpload` microservices via HTTP.
  - A storage backing service which can be one of the below:
    - `Minio` via HTTP if the `VideoStorage` microservice is built using the `MinioStorage` VSCode project.
    - `Azure Storage` via HTTP if the `VideoStorage` microservice is built using the `AzureStorage` VSCode project.
- `VideoUpload` communicates with:
  - The `Gateway` and `VideoStorage` microservices via HTTP.
  - A `RabbitMQ` backing service via AMQP, where it publishes `VideoUploaded` messages to an Exchange called `uploaded`.
- `VideoStreaming` communicates with:
  - The `Gateway` and `VideoStorage` microservices via HTTP.
  - A `RabbitMQ` backing service via AMQP, where it publishes `VideoViewed` messages to an Exchange called `viewed`.
- `Metadata` communicates with:
  - The `Gateway` microservice via HTTP.
  - An `SQLServer` backing service via HTTP, where it manages video metadata in a `Videos` table. 
  - A `RabbitMQ` backing service via AMQP, where it subscribes to an Exchange called `uploaded` receiving `VideoUploaded` messages.
- `History` communicates with:
  - The `Gateway` microservice via HTTP.
  - An `SQLServer` backing service via HTTP, where it manages video viewing history in a `ViewHistorys` table. 
  - A `RabbitMQ` backing service via AMQP, where it subscribes to an Exchange called `viewed` receiving `VideoViewed` messages.

---

## Installing a LoadBalancer

A LoadBalancer is a Kubernetes Service that exposes an *internal* Kubernetes application to the *outside world* (i.e. the *internet* when using a cloud provider) and also functions as a Layer 4 load balancer.

- When using a Minikube Kubernetes cluster, you can enable the MetalLB LoadBalancer addon.

**Note**
- The MetalLB load balancer can also be installed in a Kubernetes cluster using Helm.
  
  ```bash
  helm repo add metallb https://metallb.github.io/metallb
  helm repo update
  helm upgrade --install metallb metallb/metallb --create-namespace --namespace metallb-system
  ```

In [1]:
!minikube addons enable metallb

❗  metallb is a 3rd party addon and is not maintained or verified by minikube maintainers, enable at your own risk.
❗  metallb does not currently have an associated maintainer.
    ▪ Using image quay.io/metallb/controller:v0.9.6
    ▪ Using image quay.io/metallb/speaker:v0.9.6
🌟  The 'metallb' addon is enabled


- List the MetalLB Pods.
  - Notice the subnet (e.g. `192.168.49.x`) used by the `speaker` Pod.

In [3]:
!kubectl get pods -n metallb-system -o wide

NAME                         READY   STATUS    RESTARTS   AGE   IP             NODE       NOMINATED NODE   READINESS GATES
controller-cff57fcb5-bs6d8   1/1     Running   0          13s   10.244.0.3     minikube   <none>           <none>
speaker-rcdl9                1/1     Running   0          13s   192.168.49.2   minikube   <none>           <none>


- Deploy a ConfigMap
  - Open the file `manifests/configmap.yaml`.
  - Change the last row so that it matches your subnet.
    - E.g. if your `speaker` nodes are on subnet `192.168.49.x`, change the last row to `192.168.49.100-192.168.49.110`
    - This let's MetalLB offer `EXTERNAL-IP` addresses in that range.
  - Save the YAML file.
- Run the cell below to deploy the ConfigMap (which configures MetalLB).

In [4]:
!kubectl apply -f manifests/configmap.yaml

configmap/config configured


---

## Deploying Flixtube to Minikube's Kubernetes Cluster

### Run the command below to build the Docker images for the Flixtube application

- The command used here is `docker build -t <ImageName>:<Tag> --file <PathToDockerFile> <PathToFolderWithFiles>`
- For example, for the `Flixtube.MinioStorage` microservice:
  - `<ImageName>` is `video-storage`
  - `<Tag>` is `1`
  - `<PathToDockerFile>` relative to this notebook is `../../flixtube/Flixtube.MinioStorage/Dockerfile-prod`
  - `<PathToFolderWithFiles>` relative to this notebook is `../../flixtube/Flixtube.MinioStorage`

Note that we could replace `Dockerfile-prod` with `Dockerfile-dev` if we wanted to build the development versions of the images. 

In [5]:
# !docker build -t video-storage:1 --file ../../flixtube/Flixtube.AzureStorage/Dockerfile-prod ../../flixtube/Flixtube.AzureStorage
!docker build -t video-storage:1 --file ../../flixtube/Flixtube.MinioStorage/Dockerfile-prod ../../flixtube/Flixtube.MinioStorage
!docker build -t video-upload:1 --file ../../flixtube/Flixtube.VideoUpload/Dockerfile-prod ../../flixtube/Flixtube.VideoUpload
!docker build -t video-streaming:1 --file ../../flixtube/Flixtube.VideoStreaming/Dockerfile-prod ../../flixtube/Flixtube.VideoStreaming
!docker build -t metadata:1 --file ../../flixtube/Flixtube.Metadata/Dockerfile-prod ../../flixtube/Flixtube.Metadata
!docker build -t history:1 --file ../../flixtube/Flixtube.History/Dockerfile-prod ../../flixtube/Flixtube.History
!docker build -t gateway:1 --file ../../flixtube/Flixtube.Gateway/Dockerfile-prod ../../flixtube/Flixtube.Gateway
!docker build -t web:1 --file ../../flixtube/Flixtube.Web/Dockerfile-prod ../../flixtube/Flixtube.Web

[+] Building 0.0s (0/1)                                          docker:default
[+] Building 0.2s (1/3)                                          docker:default
 => [internal] load build definition from Dockerfile-prod                  0.1s
 => => transferring dockerfile: 449B                                       0.0s
 => [internal] load metadata for mcr.microsoft.com/dotnet/aspnet:9.0       0.2s
 => [internal] load metadata for mcr.microsoft.com/dotnet/sdk:9.0          0.2s
[+] Building 0.3s (2/3)                                          docker:default
 => [internal] load build definition from Dockerfile-prod                  0.1s
 => => transferring dockerfile: 449B                                       0.0s
 => [internal] load metadata for mcr.microsoft.com/dotnet/aspnet:9.0       0.2s
 => [internal] load metadata for mcr.microsoft.com/dotnet/sdk:9.0          0.2s
[+] Building 0.4s (4/4)                                          docker:default
 => [internal] load build definition fro

### List all Images on the Local Computer

- Notice that all the images have been built.

In [6]:
# !docker images
!docker images | grep -E 'IMAGE|web|gateway|metadata|history|video-storage|video-upload|video-streaming'

IMAGE                                                                                                 ID             DISK USAGE   CONTENT SIZE   EXTRA
gateway:1                                                                                             0e64f1cadb90        333MB         93.9MB        
history:1                                                                                             42b59fc9e132        436MB          124MB        
metadata:1                                                                                            946aaea4be8e        436MB          124MB        
video-storage:1                                                                                       961e508d7d15        332MB         93.7MB        
video-streaming:1                                                                                     f038deab7abd        330MB         93.2MB        
video-upload:1                                                                                

### Switch Kubernetes Context

- Since we are using a single-node Minikube Kubernetes cluster, run the cell below to switch to the `minikube` context.
- Note that Minikube will:
  - Automatically switch the correct context when starting the cluster.
  - Automatically switch back to the previous context when stopping the cluster.

In [7]:
!kubectl config use-context minikube

Switched to context "minikube".


### Load the Local Docker Images into Minikube Nodes

- When using a Minikube Kubernetes cluster, you need to load the local Docker images (on your host machine) into the cluster nodes.
  - This is done by using the command `minikube image load [ImageName]`, where `[ImageName]` is the name (and version) of the image.

In [8]:
!minikube image load gateway:1
!minikube image load history:1
!minikube image load metadata:1
!minikube image load video-storage:1
!minikube image load video-streaming:1
!minikube image load video-upload:1
!minikube image load web:1

- To list availabe (loaded) Docker images in the Minikube cluster, we can use the command `minikube image ls`
  - `|` pipes the output of the command `minikube image ls` as input to the command `grep probe-example`
  - `grep -i probe-example` filters its input to only output lines containing the text `probe-example` (where `-i` makes it a case-insensitive search)
- Notice that the images were loaded successfully and automatically prefixed with `docker.io/library/`.
  - This is the `default prefix` used when no explicit prefix is given, and refers to images on DockerHub.

In [9]:
# !minikube image ls
!minikube image ls | grep -E 'gateway|history|metadata|video-storage|video-streaming|video-upload|web'

docker.io/library/web:1
docker.io/library/video-upload:1
docker.io/library/video-streaming:1
docker.io/library/video-storage:1
docker.io/library/metadata:1
docker.io/library/history:1
docker.io/library/gateway:1


### Run the command below to deploy the microservices to Minikube's Kubernetes Cluster

- The main command used here is `kubectl apply -f <PathToYAMLManifestFile>`
  - For example, for the `Flixtube.MinioStorage` microservice's YAML manifest file containg its deployment and service definitions:
    - `<PathToYAMLManifestFile>` relative to this notebook is `../../flixtube/scripts/local-k8-cluster/minio.yaml`

In [10]:
!kubectl apply -f ../../flixtube/scripts/local-k8-cluster/rabbit.yaml
!kubectl apply -f ../../flixtube/scripts/local-k8-cluster/sqlserver.yaml
!kubectl apply -f ../../flixtube/scripts/local-k8-cluster/minio.yaml
!kubectl apply -f ../../flixtube/scripts/local-k8-cluster/video-storage.yaml
!kubectl apply -f ../../flixtube/scripts/local-k8-cluster/video-upload.yaml
!kubectl apply -f ../../flixtube/scripts/local-k8-cluster/video-streaming.yaml
!kubectl apply -f ../../flixtube/scripts/local-k8-cluster/metadata.yaml
!kubectl apply -f ../../flixtube/scripts/local-k8-cluster/history.yaml
!kubectl apply -f ../../flixtube/scripts/local-k8-cluster/gateway.yaml
!kubectl apply -f ../../flixtube/scripts/local-k8-cluster/web.yaml

deployment.apps/rabbit created
service/rabbit created
deployment.apps/db created
service/db created
deployment.apps/minio created
service/minio created
job.batch/minio-mc created
deployment.apps/video-storage created
service/video-storage created
deployment.apps/video-upload created
service/video-upload created
deployment.apps/video-streaming created
service/video-streaming created
deployment.apps/metadata created
service/metadata created
deployment.apps/history created
service/history created
deployment.apps/gateway created
service/gateway created
deployment.apps/web created
service/web created


### List all Pods, Deployments and Services running in the Kubernetes Cluster

- We see that all the resources have been deployed to the Kubernetes cluster.
  - Notice that the `Web` microservice is available via a `LoadBalancer` Service on port `4000`.
    - When using a Minikube Kubernetes cluster, access the `web` microservice using http://IP:4000, where `IP` is the IP address listed under `EXTERNAL-IP`, e.g. http://192.168.49.101:4000
  - Notice that the `Gateway` microservice is available via a `LoadBalancer` Service on port `4010`.
    - When using a Minikube Kubernetes cluster, access the `Gateway` microservice using http://IP:4010, where `IP` is the IP address listed under `EXTERNAL-IP`, e.g. http://192.168.49.100:4010

**Note that you might have to wait a while (30-45 seconds) before all the Pods are READY.**

In [17]:
!kubectl get pods,deployments,services -o wide

NAME                                   READY   STATUS    RESTARTS   AGE   IP            NODE       NOMINATED NODE   READINESS GATES
pod/db-57f887f5bc-m724d                1/1     Running   0          63s   10.244.0.5    minikube   <none>           <none>
pod/gateway-64645f5c7d-2s6jl           1/1     Running   0          60s   10.244.0.13   minikube   <none>           <none>
pod/history-6c946469cb-4s5rp           1/1     Running   0          60s   10.244.0.12   minikube   <none>           <none>
pod/metadata-598cd5b869-677dz          1/1     Running   0          61s   10.244.0.11   minikube   <none>           <none>
pod/minio-68976ff65-6km7s              1/1     Running   0          63s   10.244.0.6    minikube   <none>           <none>
pod/minio-mc-wsx9s                     1/1     Running   0          63s   10.244.0.7    minikube   <none>           <none>
pod/rabbit-b9f74fdfb-sj6xl             1/1     Running   0          64s   10.244.0.4    minikube   <none>           <none>
pod/vid

**Note**
- When using a Minikube Kubernetes cluster, and you see any Pods with a `STATUS` of `Error` or `CrashLoopBackOff`, run the two cells below.
  - This increases the number of `inotify watchers` (file descriptors) on the Kubernetes nodes.

In [ ]:
!minikube ssh -- "printf '%s\n' \
'fs.inotify.max_user_instances=1024' \
'fs.inotify.max_user_watches=524288' \
'fs.file-max=1000000' \
| sudo tee /etc/sysctl.d/99-inotify.conf >/dev/null && sudo sysctl --system"

* Applying /etc/sysctl.d/10-network-security.conf ...
* Applying /etc/sysctl.d/11-tcp-mtu-probing.conf ...
* Applying /usr/lib/sysctl.d/50-pid-max.conf ...
* Applying /etc/sysctl.d/99-inotify.conf ...
* Applying /usr/lib/sysctl.d/99-protect-links.conf ...
* Applying /etc/sysctl.d/99-sysctl.conf ...
* Applying /etc/sysctl.conf ...
net.ipv4.conf.default.rp_filter = 1
net.ipv4.conf.all.rp_filter = 1
net.ipv4.tcp_mtu_probing = 1
kernel.pid_max = 4194304
fs.inotify.max_user_instances = 1024
fs.inotify.max_user_watches = 524288
fs.file-max = 1000000
fs.protected_fifos = 1
fs.protected_hardlinks = 1
fs.protected_regular = 2
fs.protected_symlinks = 1


In [36]:
!minikube ssh -n minikube -- "cat /proc/sys/fs/inotify/max_user_instances /proc/sys/fs/inotify/max_user_watches /proc/sys/fs/file-max"

1024
524288
1000000


---
## Making sure the Web service talks to the Gateway service

- Let's list the manifest (YAML) we used when deploying the web service

In [18]:
#!type ../../flixtube/scripts/local-k8-cluster/web.yaml # use this on Windows
!cat ../../flixtube/scripts/local-k8-cluster/web.yaml # use this on Linux/MacOS

#
# To deploy:
#
# kubectl apply -f scripts/deploy.yaml
#
#
apiVersion: apps/v1
kind: Deployment
metadata:
  name: web
spec:
  replicas: 1
  selector:
    matchLabels:
      app: web
  template:
    metadata:
      labels:
        app: web
    spec:
      containers: 
      - name: web
        image: web:1
        imagePullPolicy: Never
        resources:
          requests:
            cpu: 100m
            memory: 128Mi
          limits:
            cpu: 250m
            memory: 256Mi
        env:
        - name: FLIXTUBE_WEB_PORT
          value: "4000"
        - name: FLIXTUBE_GATEWAY_SCHEME
          value: http
        - name: FLIXTUBE_GATEWAY_HOST
          value: gateway
        - name: FLIXTUBE_GATEWAY_PORT
          value: "4010"
        - name: FLIXTUBE_PUBLIC_GATEWAY_SCHEME
          value: http
        - name: FLIXTUBE_PUBLIC_GATEWAY_HOST
          value: localhost
        - name: FLIXTUBE_PUBLIC_GATEWAY_PORT
          value: "4010"
        ports:
        - containerPort: 

- Notice the environment variables
  - `name: FLIXTUBE_PUBLIC_GATEWAY_HOST` with `value: localhost`
  - `name: FLIXTUBE_PUBLIC_GATEWAY_PORT` with `value: "4010"`

    ```yaml
    env:
    - name: FLIXTUBE_WEB_PORT
        value: "4000"
    - name: FLIXTUBE_GATEWAY_SCHEME
        value: http
    - name: FLIXTUBE_GATEWAY_HOST
        value: gateway
    - name: FLIXTUBE_GATEWAY_PORT
        value: "4010"
    - name: FLIXTUBE_PUBLIC_GATEWAY_SCHEME
        value: http
    - name: FLIXTUBE_PUBLIC_GATEWAY_HOST
        value: localhost                   # <--- we need to change "localhost" to the Gateway's public IP address (EXTERNAL-IP)
    - name: FLIXTUBE_PUBLIC_GATEWAY_PORT
        value: "4010"
    ```

- The `web` service's environment variable `FLIXTUBE_PUBLIC_GATEWAY_HOST`:
  - Expects an IP address (`EXTERNAL-IP` below) or a DNS name.
- Let's see what `EXTERNAL-IP` the `LoadBalancer` service for the `gateway` is using by running the cell below.

In [19]:
!kubectl get service/gateway -o wide

NAME      TYPE           CLUSTER-IP       EXTERNAL-IP      PORT(S)          AGE   SELECTOR
gateway   LoadBalancer   10.110.119.165   192.168.49.100   4010:31478/TCP   94s   app=gateway


- Let's change the environment variable in the `web` deployment to the below.
  - Use the `EXTERNAL-IP` in the cell above, e.g. `192.168.49.100`:

    ```yaml
    - name: FLIXTUBE_PUBLIC_GATEWAY_HOST
        value: 192.168.49.100
    ```

In [20]:
# Change the IP address, if needed, before running the following command
!kubectl set env deployment/web FLIXTUBE_PUBLIC_GATEWAY_HOST=192.168.49.100

deployment.apps/web env updated


- Let's make sure the environment variable was changed.

In [21]:
!kubectl rollout status deployment/web
!kubectl exec deploy/web -- printenv | grep FLIXTUBE_PUBLIC_GATEWAY

deployment "web" successfully rolled out
FLIXTUBE_PUBLIC_GATEWAY_SCHEME=http
FLIXTUBE_PUBLIC_GATEWAY_HOST=192.168.49.100
FLIXTUBE_PUBLIC_GATEWAY_PORT=4010


---

## Using the Flixtube Application

- Visit the Home page:
  - The URL is http://IP:4000, where `IP` is your LoadBalancer service's IP for your `web` microservice, e.g. http://192.168.49.101:4000
  - The Home page displays a list of videos (currently empty) with an `Id` and a `Name` column.
  - There are two buttons `Upload Video` and `Show Viewing History` at the top of the page.
    - The `Upload Video` button is used for uploading a video.
    - The `Show Viewing History` button is used to display a list of viewing history, i.e. when a video was viewed (played).
  - Click the `Upload Video` button on the Home page which will navigate to the Upload Video page.

<img src="notebook_images/flixtube_home_page.png"  width="600" alt="Flixtube Home Page" />

- The Upload Video page.

  - The Upload Video page allows the user to upload a video from the user's local file system.
  - There are four buttons `Home`, `Choose File`, `Upload` and `Cancel` on the page.
    - The `Home` button navigates back to the Home page.
    - The `Choose File` button pops up a window, allowing the user to choose a video file to upload.
      - The name of the chosen video file is displayed in the field to the right of the `Choose File` button.
    - The `Upload` button uploads the chosen video file.
    - The `Cancel` button returns the user to the Home page without uploading any video file.
  - Click the `Choose File` button and choose a video file.
    - There is a sample file in `workshop4/05_flixtube/SampleVideo_1280x720_1mb.mp4` you can use.
  - Then click the `Upload` button, which will upload the file and navigate back to the Home page.

<img src="notebook_images/flixtube_upload_page.png"  width="600" alt="Flixtube Upload Video Page" />

- Back on the Home page.
  - The Home page now displays one video in its a list of videos.
    - The `Id` column contains the video's GUID.
    - The `Name` column contains the video's name.
    - The blue `Play` button is used to play (view) the video.
    - The red `Delete` button is used to remove (delete) the video.
  - We could also access the Gateway to get the JSON document via the REST API.
    - The URL is http://IP:4010/api/metadata, where `IP` is your LoadBalancer's IP, e.g. http://192.168.49.100:4010/api/metadata
- Click the blue `Play` button which will navigate to the Play Video page.

<img src="notebook_images/flixtube_home_page2.png"  width="600" alt="Flixtube Home Page" />

- The Play Video page.

  - The Play Video page starts streaming the video to the browser.
    - The video's name is shown above the video.
    - Video controls for playing, pausing, etc. are shown below the video.
    - There is a `Home` button at the top of the page, which will navigate back to the Home page.
  - Click the `Home` button to navigate back to the Home page.

<img src="notebook_images/flixtube_play_page.png"  width="600" alt="Flixtube Play Video Page" />

- Back on the Home page.

  - Click the `Show Viewing History` button which will navigate to the Viewing History page.

<img src="notebook_images/flixtube_home_page2.png"  width="600" alt="Flixtube Home Page" />

- The Viewing History page.
  - The Viewing History page displays a list video viewings with an `Id` and a `ViewedAt` column.
    - The `Id` column contains the video's GUID.
    - The `ViewedAt` column contains the date and time the video was viewed (played).
    - There is a `Home` button at the top of the page, which will navigate back to the Home page.
  - We could also access the Gateway to get the JSON document via the REST API.
    - The URL is http://IP:4010/api/history, where `IP` is your LoadBalancer service's IP for your `gateway` microservice, e.g. http://192.168.49.100:4010/api/history
  - Click the `Home` button to navigate back to the Home page.

<img src="notebook_images/flixtube_history_page.png"  width="600" alt="Flixtube Viewing History Page" />

---

## Remove Flixtube from Minikube's Kubernetes Cluster

### Run the command below to delete the microservices from Minikube's Kubernetes Cluster

- The main command used here is `kubectl delete -f <PathToYAMLManifestFile>`
  - For example, for the `Flixtube.MinioStorage` microservice's YAML manifest file containg its deployment and service definitions:
    - `<PathToYAMLManifestFile>` relative to this notebook is `../../flixtube/scripts/local-k8-cluster/minio.yaml`

In [22]:
!kubectl delete -f ../../flixtube/scripts/local-k8-cluster/rabbit.yaml
!kubectl delete -f ../../flixtube/scripts/local-k8-cluster/sqlserver.yaml
!kubectl delete -f ../../flixtube/scripts/local-k8-cluster/minio.yaml
!kubectl delete -f ../../flixtube/scripts/local-k8-cluster/video-storage.yaml
!kubectl delete -f ../../flixtube/scripts/local-k8-cluster/video-upload.yaml
!kubectl delete -f ../../flixtube/scripts/local-k8-cluster/video-streaming.yaml
!kubectl delete -f ../../flixtube/scripts/local-k8-cluster/metadata.yaml
!kubectl delete -f ../../flixtube/scripts/local-k8-cluster/history.yaml
!kubectl delete -f ../../flixtube/scripts/local-k8-cluster/gateway.yaml
!kubectl delete -f ../../flixtube/scripts/local-k8-cluster/web.yaml

deployment.apps "rabbit" deleted from default namespace
service "rabbit" deleted from default namespace
deployment.apps "db" deleted from default namespace
service "db" deleted from default namespace
deployment.apps "minio" deleted from default namespace
service "minio" deleted from default namespace
job.batch "minio-mc" deleted from default namespace
deployment.apps "video-storage" deleted from default namespace
service "video-storage" deleted from default namespace
deployment.apps "video-upload" deleted from default namespace
service "video-upload" deleted from default namespace
deployment.apps "video-streaming" deleted from default namespace
service "video-streaming" deleted from default namespace
deployment.apps "metadata" deleted from default namespace
service "metadata" deleted from default namespace
deployment.apps "history" deleted from default namespace
service "history" deleted from default namespace
deployment.apps "gateway" deleted from default namespace
service "gateway" d

### List all Pods, Deployments and Services running in Minikube's Kubernetes Cluster

- We see that all the resources have been removed from the Kubernetes cluster.
  - The only service displayed in the list should be the `service/kubernetes` service which we should never delete.

**Note that it might take 30-45 seconds before all the Flixtube resources have been removed from the cluster.**

In [28]:
!kubectl get pods,deployments,services -o wide

NAME                 TYPE        CLUSTER-IP   EXTERNAL-IP   PORT(S)   AGE     SELECTOR
service/kubernetes   ClusterIP   10.96.0.1    <none>        443/TCP   8m10s   <none>


### Run the command below to remove the Docker images for the Flixtube application

- The command used here is `docker rmi <ImageName>:<Tag> --force`
- For example, for the `Flixtube.MinioStorage` microservice:
  - `<ImageName>` is `video-storage`
  - `<Tag>` is `1`

In [29]:
!docker rmi video-storage:1 --force
!docker rmi video-upload:1 --force
!docker rmi video-streaming:1 --force
!docker rmi metadata:1 --force
!docker rmi history:1 --force
!docker rmi gateway:1 --force
!docker rmi web:1 --force

Untagged: video-storage:1
Deleted: sha256:961e508d7d1504581b42b97e35210ab92f173f7d7a03cad9d108dd895c3e25ee
Untagged: video-upload:1
Deleted: sha256:ab33d770d2ec1bd8f3315bbb178e876b18e8c2113427908669670ffdb9f85e1b
Untagged: video-streaming:1
Deleted: sha256:f038deab7abd38a89e2b1e47c8357e56b9c2dec401feb6361bba5b06053f1c47
Untagged: metadata:1
Deleted: sha256:946aaea4be8e527fae29340bc1a05f7539a1dafe6eb2ef99219170c62a36430a
Untagged: history:1
Deleted: sha256:42b59fc9e1326e1cf06c4b748677942a71ec2943a33843454f2f3870ac336f49
Untagged: gateway:1
Deleted: sha256:0e64f1cadb906a8786d232fad7164937efa4227f96edc661e74e7f7096ec89b9
Untagged: web:1
Deleted: sha256:268025ac0a8294b47598c39ac8cf4508af66545a0558be81faa911b7ec63078d


### List all Images on the Local Computer

- Notice that all the images have been removed.

In [30]:
# !docker images
!docker images | grep -E 'web|gateway|metadata|history|video-storage|video-upload|video-streaming'